Building Advanced RAG Q&A Project With Multiple Data Sources With Langchain

In [1]:
from langchain_community.tools import WikipediaQueryRun 
from langchain_community.utilities import WikipediaAPIWrapper

API wrapper
* this is basically use to query answer from wikipedia

In [3]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [4]:
wiki.name

'wikipedia'

In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY') 

In [11]:
loader=WebBaseLoader('https://docs.smith.langchain.com/')
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

In [13]:
## creating vector DB
from langchain_huggingface import HuggingFaceEmbeddings
vectordb=FAISS.from_documents(documents, HuggingFaceEmbeddings())
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002CC2956DDD0>, search_kwargs={})

In [15]:
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith_search","Search for information about Langsmith. For any questions about LangSmith, youmust use this tool")

In [16]:
retriever_tool.name

'langsmith_search'

In [17]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

In [19]:
arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

Combine All Tools

In [20]:
tools=[wiki,arxiv,retriever_tool]

In [21]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\D_Drive\\FullStackGenai\\fsgenai\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='langsmith_search', description='Search for information about Langsmith. For any questions about LangSmith, youmust use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000002CC299659E0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000002CC29965E40>)]

Agents
* The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order. Learn more

In [39]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [37]:

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

In [40]:
from langchain_core.output_parsers import StrOutputParser
# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({
    "input": "Explain Agentic AI"
})

print(response)


Agentic AI refers to a type of artificial intelligence (AI) that is designed to act autonomously, making decisions and taking actions on its own, without explicit human intervention. The term "agentic" comes from the word "agent," which in AI refers to a system that can perceive its environment, reason about it, and take actions to achieve its goals.

Agentic AI systems are characterized by the following key features:

1. **Autonomy**: Agentic AI systems can operate independently, making decisions and taking actions without human oversight or control.
2. **Goal-oriented**: Agentic AI systems have specific goals or objectives that they strive to achieve, and they can adapt their behavior to achieve those goals.
3. **Self-awareness**: Agentic AI systems have a sense of their own capabilities, limitations, and context, which enables them to make informed decisions.
4. **Learning and adaptation**: Agentic AI systems can learn from their experiences and adapt to changing environments, allow